In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [3]:
print("--- 🚀 STARTING PIPELINE: Cleaning, Validation & Analysis ---")

# ======================================================
# 1. LOAD DATASETS
# ======================================================
SCENARIO_FILE = 'Trump_Tariff_Dataset 2.csv'
PROFILE_FILE = 'Trump_Tariff_Dataset 1.csv'

try:
    df_scenarios_raw = pd.read_csv(SCENARIO_FILE, encoding='latin-1')
    df_profiles_raw = pd.read_csv(PROFILE_FILE, encoding='latin-1')
    print("✅ Files Loaded Successfully.")
except FileNotFoundError:
    print("❌ Error: Files not found. Please check filenames.")

--- 🚀 STARTING PIPELINE: Cleaning, Validation & Analysis ---
✅ Files Loaded Successfully.


In [4]:
# ======================================================
# 2. CLEAN PROFILE DATA (Dataset 1)
# ======================================================
print("\n--- Cleaning Profiles (Deficits & Populations) ---")

cols_to_clean = ['US 2024 Deficit', 'US 2024 Exports', 'US 2024 Imports (Customs Basis)', 'Population']
for col in cols_to_clean:
    if col in df_profiles_raw.columns:
        df_profiles_raw[col] = df_profiles_raw[col].astype(str).str.replace(',', '', regex=False)
        df_profiles_raw[col] = df_profiles_raw[col].astype(str).str.replace('"', '', regex=False)
        df_profiles_raw[col] = pd.to_numeric(df_profiles_raw[col], errors='coerce').fillna(0)

pct_cols = ['Trump Tariffs Alleged', 'Trump Tarrif Response']
for col in pct_cols:
    if col in df_profiles_raw.columns:
        df_profiles_raw[col] = df_profiles_raw[col].astype(str).str.replace('%', '', regex=False)
        df_profiles_raw[col] = pd.to_numeric(df_profiles_raw[col], errors='coerce') / 100

if 'Country' in df_profiles_raw.columns:
    df_profiles_raw = df_profiles_raw.rename(columns={'Country': 'Countries'})

print(f"   -> Profiles Cleaned: {len(df_profiles_raw)} countries processed.")


--- Cleaning Profiles (Deficits & Populations) ---
   -> Profiles Cleaned: 204 countries processed.


In [5]:
# ======================================================
# 3. CLEAN SCENARIO DATA (Dataset 2)
# ======================================================
print("\n--- Cleaning Scenarios (Time Series) ---")

# Fix Column Names (Remove hidden characters)
df_scenarios_raw.columns = df_scenarios_raw.columns.str.strip().str.replace('ï»¿', '')

if 'date' not in df_scenarios_raw.columns:
    for col in df_scenarios_raw.columns:
        if 'date' in col.lower():
            df_scenarios_raw = df_scenarios_raw.rename(columns={col: 'date'})
            break

df_scenarios = df_scenarios_raw.dropna(subset=['date', 'Countries'], how='all').copy()
df_scenarios['date'] = pd.to_datetime(df_scenarios['date'], errors='coerce')
df_scenarios = df_scenarios.dropna(subset=['date'])

def clean_tariff_text(val):
    if pd.isna(val): return np.nan
    text = str(val)
    matches = re.findall(r"(\d+(?:\.\d+)?)", text)
    if matches:
        num = float(matches[0])
        if num > 1: return num / 100.0
        return num
    return np.nan

df_scenarios['TarrifImpose_Clean'] = df_scenarios['TarrifImpose'].apply(clean_tariff_text)
df_scenarios['avEffectiveTariffRate'] = df_scenarios['avEffectiveTariffRate'].fillna(df_scenarios['TarrifImpose_Clean'])

print(f"   -> Scenarios Cleaned: {len(df_scenarios)} valid events found.")


--- Cleaning Scenarios (Time Series) ---
   -> Scenarios Cleaned: 57 valid events found.


In [8]:
# ======================================================
# 4. MERGE, VALIDATE & TRAIN
# ======================================================
print("\n--- Merging Data & Training Random Forest ---")

# Fix Column Names in df_profiles_raw (similar to df_scenarios_raw cleaning in cell kVS3ySsgbxnU)
df_profiles_raw.columns = df_profiles_raw.columns.str.strip().str.replace('ï»¿', '')
if 'Country' in df_profiles_raw.columns:
    df_profiles_raw = df_profiles_raw.rename(columns={'Country': 'Countries'})

df_viz = pd.merge(df_scenarios, df_profiles_raw[['Countries', 'US 2024 Deficit']], on='Countries', how='left')

# Prepare Training Data
train_cols = ['gdp', 'cpeInflation', 'avEffectiveTariffRate', 'US 2024 Deficit']
train_df = df_viz.dropna(subset=train_cols)

if len(train_df) > 5:
    X = train_df[['avEffectiveTariffRate', 'US 2024 Deficit']]
    y_gdp = train_df['gdp']
    y_inf = train_df['cpeInflation']

    # --- Ÿ VALIDATION STEP (BACK TESTING) ---
    print("\n[Validation] Performing Train/Test Split & Calculating RMSE...")
    # 1. Split Data (80% Train, 20% Test)
    X_train, X_test, y_train, y_test = train_test_split(X, y_gdp, test_size=0.2, random_state=42)

    # 2. Train Validation Model
    rf_val = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)

    # 3. Predict & Calculate Error
    y_pred_val = rf_val.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_val))

    print(f"\u2705 Model Validation Score (RMSE): {rmse:.5f}")
    print("   ✉ SCREENSHOT THIS SCORE FOR YOUR THESIS!")
    # ------------------------------------------

    # Full Model Training (For Final Dashboard)
    rf_gdp = RandomForestRegressor(n_estimators=100, random_state=42).fit(X, y_gdp)
    rf_inf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X, y_inf)
    print("\u2705 Final Model Trained on Full Dataset.")

    # Fill Predictions
    X_all = df_viz[['avEffectiveTariffRate', 'US 2024 Deficit']].fillna(0)
    df_viz['gdp'] = df_viz['gdp'].fillna(pd.Series(rf_gdp.predict(X_all), index=df_viz.index))
    df_viz['cpeInflation'] = df_viz['cpeInflation'].fillna(pd.Series(rf_inf.predict(X_all), index=df_viz.index))

    # Sensitivity Calculation
    X_sim = X_all.copy()
    X_sim['avEffectiveTariffRate'] = X_sim['avEffectiveTariffRate'] + 0.01
    df_viz['GDP_Sensitivity_Factor'] = rf_gdp.predict(X_sim) - df_viz['gdp']
else:
    print("⚠️ Warning: Not enough data to train model. Sensitivity set to 0.")
    df_viz['GDP_Sensitivity_Factor'] = 0


--- Merging Data & Training Random Forest ---

[Validation] Performing Train/Test Split & Calculating RMSE...
✅ Model Validation Score (RMSE): 0.00486
   ✉ SCREENSHOT THIS SCORE FOR YOUR THESIS!
✅ Final Model Trained on Full Dataset.


In [9]:
# ======================================================
# 5. APPLY BUSINESS RULES
# ======================================================
print("\n--- Applying 'Triple Risk' Logic ---")

def generate_insight(row):
    is_high_tariff = row['avEffectiveTariffRate'] > 0.15
    is_high_deficit = abs(row['US 2024 Deficit']) > 50000

    inf_score = min(max(row['cpeInflation'], 0) * 1000, 100)
    gdp_score = min(abs(min(row['gdp'], 0)) * 2000, 100)
    base_score = (0.6 * inf_score) + (0.4 * gdp_score)

    multiplier = 3.0 if (is_high_tariff and is_high_deficit) else 1.0
    final_score = min(base_score * multiplier, 100)

    if final_score > 75: label = 'CRITICAL'
    elif final_score > 40: label = 'HIGH'
    elif final_score > 20: label = 'MEDIUM'
    else: label = 'LOW'

    if label == 'CRITICAL':
        insight = f"CRITICAL ALERT: High Tariff (>15%) + Large Deficit (>${abs(row['US 2024 Deficit'])/1000:.0f}B). Risk TRIPLED. Immediate exit advised."
    elif label == 'HIGH':
        insight = f"High Risk. Costs rising ({row['cpeInflation']:.2%}). Monitor closely."
    elif label == 'LOW':
        insight = f"Safe Haven Candidate. GDP Stable ({row['gdp']:.2%})."
    else:
        insight = "Moderate Risk. Monitor situation."

    return pd.Series([final_score, label, insight])

df_viz[['Sourcing_Risk_Score', 'Risk_Label', 'Actionable_Insight']] = df_viz.apply(generate_insight, axis=1)


--- Applying 'Triple Risk' Logic ---


In [10]:
output_file = 'FINAL_PROJECT_DASHBOARD_DATA.csv'
cols = ['date', 'Countries', 'TarrifImpose_Clean', 'avEffectiveTariffRate', 'US 2024 Deficit',
        'gdp', 'cpeInflation', 'GDP_Sensitivity_Factor',
        'Sourcing_Risk_Score', 'Risk_Label', 'Actionable_Insight']

df_viz[cols].to_csv(output_file, index=False)
print(f"\n✅ DONE! Final file saved as: {output_file}")


✅ DONE! Final file saved as: FINAL_PROJECT_DASHBOARD_DATA.csv
